# Audio V5 Sampling Playground

Cell 1 sets up helpers. Cell 2 is the single block to tweak and run.

In [11]:
from pathlib import Path
import os
import pickle
import warnings
from contextlib import nullcontext

import torch
from IPython.display import Audio, display

from audio_acoustic_decoder import AudioAcousticDecoder, AudioAcousticDecoderConfig, decode_codec_frames_from_semantics
from audio_codec import decode_codes, encode_waveform, load_audio, load_encodec_model, save_waveform
from audio_prosody import extract_prosody_features
from audio_semantic_model import AudioSemanticGPT, AudioSemanticGPTConfig, generate_semantic_tokens
from audio_semantic_tokenizer import SemanticTokenizer

# ignore all warnings
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
MODEL_CACHE = {}
print(ROOT)


def maybe_trim_waveform(wav, sample_rate, max_seconds):
    if max_seconds <= 0:
        return wav
    max_samples = int(sample_rate * max_seconds)
    if wav.size(-1) <= max_samples:
        return wav
    return wav[..., :max_samples]


def checkpoint_signature(out_dir):
    ckpt_path = Path(out_dir) / "ckpt.pt"
    stat = ckpt_path.stat()
    return str(ckpt_path.resolve()), stat.st_mtime_ns, stat.st_size


def load_model_cached(model_cls, config_cls, out_dir, *, device, compile_model, force_reload=False):
    sig = checkpoint_signature(out_dir)
    cache_key = (model_cls.__name__, sig, device, bool(compile_model))
    if force_reload:
        MODEL_CACHE.pop(cache_key, None)
    if cache_key in MODEL_CACHE:
        return MODEL_CACHE[cache_key]

    checkpoint = torch.load(sig[0], map_location=device)
    model = model_cls(config_cls(**checkpoint["model_args"]))
    state_dict = checkpoint["model"]
    unwanted_prefix = "_orig_mod."
    for key in list(state_dict.keys()):
        if key.startswith(unwanted_prefix):
            state_dict[key[len(unwanted_prefix):]] = state_dict.pop(key)
    model.load_state_dict(state_dict)
    model.eval()
    model.to(device)
    if compile_model:
        model = torch.compile(model)
    MODEL_CACHE[cache_key] = (model, checkpoint)
    return MODEL_CACHE[cache_key]


def describe_model_context(semantic_out_dir, acoustic_out_dir, *, prompt_max_seconds=None, max_new_seconds=None):
    semantic_ckpt = torch.load(Path(semantic_out_dir) / "ckpt.pt", map_location="cpu")
    acoustic_ckpt = torch.load(Path(acoustic_out_dir) / "ckpt.pt", map_location="cpu")

    semantic_dataset = semantic_ckpt.get("config", {}).get("dataset", "audio_semantic_codec")
    acoustic_dataset = acoustic_ckpt.get("config", {}).get("dataset", semantic_dataset)
    if semantic_dataset != acoustic_dataset:
        raise ValueError(f"Dataset mismatch: semantic={semantic_dataset!r}, acoustic={acoustic_dataset!r}")

    meta = pickle.load(open(Path("data") / semantic_dataset / "meta.pkl", "rb"))
    semantic_rate = float(meta["semantic_rate_hz_estimate"])
    codec_rate = float(meta["codec_frame_rate_hz_estimate"])
    style_prompt_frames = int(acoustic_ckpt["model_args"].get("style_prompt_frames", 0))
    semantic_block = int(semantic_ckpt["model_args"]["block_size"])
    acoustic_block = int(acoustic_ckpt["model_args"]["block_size"])
    semantic_context_seconds = semantic_block / semantic_rate
    acoustic_context_seconds = acoustic_block / codec_rate
    style_prompt_seconds = style_prompt_frames / codec_rate if style_prompt_frames else 0.0
    requested_prompt_seconds = float(prompt_max_seconds) if prompt_max_seconds is not None else None
    requested_new_seconds = float(max_new_seconds) if max_new_seconds is not None else None
    return {
        "dataset": semantic_dataset,
        "semantic_block_size": semantic_block,
        "semantic_rate_hz": semantic_rate,
        "semantic_context_seconds": semantic_context_seconds,
        "acoustic_block_size": acoustic_block,
        "codec_frame_rate_hz": codec_rate,
        "acoustic_context_seconds": acoustic_context_seconds,
        "style_prompt_frames": style_prompt_frames,
        "style_prompt_seconds": style_prompt_seconds,
        "requested_prompt_seconds": requested_prompt_seconds,
        "requested_new_seconds": requested_new_seconds,
        "semantic_context_note": "semantic model conditions on a sliding window this long",
        "acoustic_context_note": "decoder keeps only this much recent generated audio in its rolling context",
        "style_prompt_note": "decoder also gets this much prompt audio explicitly for style/prosody conditioning",
        "semantic_prompt_truncates": (requested_prompt_seconds is not None and requested_prompt_seconds > semantic_context_seconds),
        "acoustic_generation_exceeds_window": (requested_new_seconds is not None and requested_new_seconds > acoustic_context_seconds),
        "acoustic_generation_behavior": (
            "if requested_new_seconds exceeds acoustic_context_seconds, generation still continues, but the decoder only remembers a sliding tail of recent generated frames"
            if requested_new_seconds is not None else "set max_new_seconds to evaluate sliding-window behavior"
        ),
    }


def sample_v5(
    semantic_out_dir,
    acoustic_out_dir,
    prompt_audio,
    output_wav,
    *,
    prompt_max_seconds=1.0,
    max_new_seconds=1.5,
    semantic_source="predicted",
    temperature=0.9,
    top_k=100,
    seed=1337,
    device="cuda",
    dtype="float16",
    compile_model=False,
    force_reload=False,
    clear_cache=False,
):
    if clear_cache:
        MODEL_CACHE.clear()

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    ptdtype = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[dtype]
    device_type = "cuda" if "cuda" in device else "cpu"
    ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

    semantic_model, semantic_checkpoint = load_model_cached(
        AudioSemanticGPT,
        AudioSemanticGPTConfig,
        semantic_out_dir,
        device=device,
        compile_model=compile_model,
        force_reload=force_reload,
    )
    acoustic_model, acoustic_checkpoint = load_model_cached(
        AudioAcousticDecoder,
        AudioAcousticDecoderConfig,
        acoustic_out_dir,
        device=device,
        compile_model=compile_model,
        force_reload=force_reload,
    )

    semantic_dataset = semantic_checkpoint.get("config", {}).get("dataset", "audio_semantic_codec")
    acoustic_dataset = acoustic_checkpoint.get("config", {}).get("dataset", semantic_dataset)
    if semantic_dataset != acoustic_dataset:
        raise ValueError(f"Dataset mismatch: semantic={semantic_dataset!r}, acoustic={acoustic_dataset!r}")

    meta_path = os.path.join("data", semantic_dataset, "meta.pkl")
    with open(meta_path, "rb") as handle:
        meta = pickle.load(handle)

    semantic_tokenizer = SemanticTokenizer.from_metadata(meta["semantic_tokenizer"])
    codec = load_encodec_model(
        model_name=meta["codec_model_name"],
        bandwidth=meta["bandwidth"],
        device="cpu",
    )

    full_wav, sample_rate = load_audio(prompt_audio)
    prompt_wav = maybe_trim_waveform(full_wav, sample_rate, prompt_max_seconds)
    semantic_batch_full = semantic_tokenizer.encode_waveform(full_wav, sample_rate)
    semantic_batch = semantic_tokenizer.encode_waveform(prompt_wav, sample_rate)
    codec_batch = encode_waveform(codec, prompt_wav, sample_rate, device="cpu", codebook_size=meta["codebook_size"])
    prompt_prosody = extract_prosody_features(
        semantic_batch.normalized_wav if semantic_batch.normalized_wav is not None else prompt_wav,
        meta["sample_rate"],
        target_frames=codec_batch.codes.size(1),
        frame_rate_hz=float(meta["codec_frame_rate_hz_estimate"]),
    )

    semantic_prompt_cpu = semantic_batch.tokens.to(torch.long)[None, :]
    semantic_prompt = semantic_prompt_cpu.to(device)
    codec_frame_rate = float(meta["codec_frame_rate_hz_estimate"])
    semantic_rate = float(meta["semantic_rate_hz_estimate"])
    max_new_semantic = max(1, int(round(max_new_seconds * semantic_rate)))

    with torch.no_grad():
        with ctx:
            if semantic_source == "predicted":
                generated_semantic = generate_semantic_tokens(
                    semantic_model,
                    semantic_prompt,
                    max_new_tokens=max_new_semantic,
                    temperature=temperature,
                    top_k=top_k,
                )
            elif semantic_source == "ground_truth":
                full_semantic = semantic_batch_full.tokens.to(torch.long)
                prompt_semantic_len = semantic_batch.tokens.numel()
                future_semantic = full_semantic[prompt_semantic_len:prompt_semantic_len + max_new_semantic]
                if future_semantic.numel() == 0:
                    raise ValueError("No future semantic tokens available. Use a shorter prompt or longer source clip.")
                generated_semantic = torch.cat((semantic_prompt_cpu, future_semantic[None, :]), dim=1).to(device)
            else:
                raise ValueError(f"Unsupported semantic_source={semantic_source!r}")

            decoded_frames = decode_codec_frames_from_semantics(
                acoustic_model,
                generated_semantic,
                prompt_codec_frames=codec_batch.codes.transpose(0, 1).contiguous()[None, ...].to(device),
                prompt_prosody_features=prompt_prosody[None, ...].to(device),
                target_frames=int(round(max_new_seconds * codec_frame_rate)),
                temperature=temperature,
                top_k=top_k,
            )

    continuation_codes = decoded_frames[0].transpose(0, 1).contiguous().cpu()
    full_codes = torch.cat((codec_batch.codes, continuation_codes), dim=1)
    generated_wav = decode_codes(codec, full_codes, device="cpu")
    prompt_wav_out = codec_batch.normalized_wav if codec_batch.normalized_wav is not None else prompt_wav
    continuation_wav = decode_codes(codec, continuation_codes, device="cpu")

    os.makedirs(os.path.dirname(output_wav), exist_ok=True)
    root, ext = os.path.splitext(output_wav)
    prompt_path = f"{root}_prompt{ext}"
    continuation_path = f"{root}_continuation{ext}"
    save_waveform(prompt_path, prompt_wav_out, meta["sample_rate"])
    save_waveform(output_wav, generated_wav, meta["sample_rate"])
    save_waveform(continuation_path, continuation_wav, meta["sample_rate"])

    return {
        "prompt_path": prompt_path,
        "output_path": output_wav,
        "continuation_path": continuation_path,
        "semantic_dataset": semantic_dataset,
        "generated_semantic_tokens": int(generated_semantic.size(1)),
    }

/home/jd/projects/aiplayground/nanoGPT


In [15]:
import json
# Single tweak-and-run block.
# Edit this cell, then run it. It will sample and display the audio inline.

semantic_out_dir = "out-audio-semantic-v5-hubert"
acoustic_out_dir = "out-audio-acoustic-decoder-v5-hubert"
prompt_audio = "data/audio_librispeech_codec/raw/LibriSpeech/dev-clean-2/174/168635/174-168635-0002.flac"
output_wav = "out-audio-v5-hubert/notebook_generated.wav"

prompt_max_seconds = 15.0
max_new_seconds = 15.0
semantic_source = "ground_truth"   # or 'ground_truth'
temperature = 0.8
top_k = 100
seed = 1337

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else ("float16" if torch.cuda.is_available() else "float32")
compile_model = False

reload_models = False      # set True if you changed checkpoints and want to force reload
clear_model_cache = False  # set True to drop all cached models before running

context_info = describe_model_context(
    semantic_out_dir,
    acoustic_out_dir,
    prompt_max_seconds=prompt_max_seconds,
    max_new_seconds=max_new_seconds,
)
print(json.dumps(context_info, indent=2))

result = sample_v5(
    semantic_out_dir=semantic_out_dir,
    acoustic_out_dir=acoustic_out_dir,
    prompt_audio=prompt_audio,
    output_wav=output_wav,
    prompt_max_seconds=prompt_max_seconds,
    max_new_seconds=max_new_seconds,
    semantic_source=semantic_source,
    temperature=temperature,
    top_k=top_k,
    seed=seed,
    device=device,
    dtype=dtype,
    compile_model=compile_model,
    force_reload=reload_models,
    clear_cache=clear_model_cache,
)

print(json.dumps(result, indent=2))
print("Continuation Only")
display(Audio(result["continuation_path"]))
print("Prompt")
display(Audio(result["prompt_path"]))
print("Full Output")
display(Audio(result["output_path"]))

{
  "dataset": "audio_semantic_codec_hubert",
  "semantic_block_size": 256,
  "semantic_rate_hz": 24.996413918848273,
  "semantic_context_seconds": 10.241469069567856,
  "acoustic_block_size": 384,
  "codec_frame_rate_hz": 75.0,
  "acoustic_context_seconds": 5.12,
  "style_prompt_frames": 96,
  "style_prompt_seconds": 1.28,
  "requested_prompt_seconds": 15.0,
  "requested_new_seconds": 15.0,
  "semantic_context_note": "semantic model conditions on a sliding window this long",
  "acoustic_context_note": "decoder keeps only this much recent generated audio in its rolling context",
  "style_prompt_note": "decoder also gets this much prompt audio explicitly for style/prosody conditioning",
  "semantic_prompt_truncates": true,
  "acoustic_generation_exceeds_window": true,
  "acoustic_generation_behavior": "if requested_new_seconds exceeds acoustic_context_seconds, generation still continues, but the decoder only remembers a sliding tail of recent generated frames"
}
{
  "prompt_path": "out-

Prompt


Full Output
